# Analyse profonde — Matrices d'affinité successives

**Modifiez uniquement le bloc Paramètres ci-dessous**, puis exécutez toutes les cellules (`Kernel → Restart & Run All`).

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PARAMÈTRES  —  seul bloc à modifier
# ═══════════════════════════════════════════════════════════════
FIXED        = [26, 46]   # deux numéros fixés
KIND         = 'euro'     # 'loto' | 'euro' | 'all'
K_SEEDS      = 3          # tirages-graines pour la méthode filtrée
N_TOP        = 12         # compagnons affichés par heatmap
MIN_AFFINITY = 1          # seuil d'affinité brut (ignoré si METRIC='lift')
METRIC       = 'lift'     # 'lift' (corrigé) | 'count' (brut)
# ═══════════════════════════════════════════════════════════════

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from IPython.display import display

import euroloto
from euroloto._analyzer import cooccurrence_matrix
from euroloto._config import GAMES

euroloto.init('../config.yaml')
sns.set_theme(style='whitegrid', font_scale=0.9)

---
## 1. Matrice globale de co-occurrences

Vue d'ensemble : combien de fois chaque paire de numéros est sortie ensemble sur l'ensemble des tirages historiques.  
Les zones orange/rouge indiquent une forte association globale.

In [ ]:
_kind_main = 'euro' if KIND == 'all' else KIND
_df_main   = euroloto.draws(_kind_main)
_cfg_main  = GAMES[_kind_main]
_cols_main = _cfg_main['main_cols']
n_min, n_max = _cfg_main['main_range']

_cooc_full = cooccurrence_matrix(_df_main, _cols_main)
_nums_full = list(range(n_min, n_max + 1))
_mat_full  = _cooc_full.reindex(_nums_full).reindex(_nums_full, axis=1).fillna(0).astype(float)
np.fill_diagonal(_mat_full.values, np.nan)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    _mat_full, cmap='YlOrRd', ax=ax, annot=False, linewidths=0,
    xticklabels=_nums_full, yticklabels=_nums_full,
    cbar_kws={'label': 'Co-occurrences', 'shrink': 0.7},
    mask=np.isnan(_mat_full),
)
for n in FIXED:
    if n_min <= n <= n_max:
        idx = n - n_min
        ax.axhline(idx,     color='#2980b9', lw=1.5, alpha=0.7)
        ax.axhline(idx + 1, color='#2980b9', lw=1.5, alpha=0.7)
        ax.axvline(idx,     color='#2980b9', lw=1.5, alpha=0.7)
        ax.axvline(idx + 1, color='#2980b9', lw=1.5, alpha=0.7)

ax.set_title(
    f'{_cfg_main["name"]} — Matrice globale de co-occurrences '
    f'({len(_df_main):,} tirages)\nLignes/colonnes bleues = numéros fixés {FIXED}',
    fontsize=13, fontweight='bold',
)
ax.tick_params(labelsize=7)
fig.tight_layout()
plt.show()

---
## 2. Tableau des compagnons avec lift

`lift > 1` → le numéro apparaît **plus souvent** avec la paire fixée qu'attendu par le hasard  
`lift < 1` → association due au hasard, à ignorer

In [ ]:
comp_table = euroloto.companions(FIXED, kind=KIND, n_top=20)
print(f"Compagnons de {FIXED} ({KIND}) — triés par fréquence")

_st = comp_table.style
if 'lift' in comp_table.columns:          # mode loto / euro
    _fmt = {'pct': '{:.1f}%', 'lift': '{:.2f}'}
    _st = (_st
           .background_gradient(subset=['frequence'], cmap='Blues')
           .background_gradient(subset=['lift'], cmap='RdYlGn', vmin=0.5, vmax=4))
else:                                     # mode all
    _fmt = {'pct_%': '{:.1f}%'}
    _st = (_st
           .background_gradient(subset=['total'], cmap='Blues')
           .background_gradient(subset=['loto'],  cmap='PuBu')
           .background_gradient(subset=['euro'],  cmap='OrRd'))
display(_st.format(_fmt))

---
## 3. Matrices d'affinité successives — 3ème, 4ème, 5ème numéro

Chaque heatmap montre **l'affinité conditionnelle** entre les candidats compagnons dans les tirages contenant l'ensemble fixé courant.  
Le meilleur candidat (fréquence max) est automatiquement sélectionné pour passer à l'étape suivante.

- **`metric='lift'`** : cellule orange = les deux numéros apparaissent ensemble significativement plus souvent qu'attendu  
- **`metric='count'`** : co-occurrence brute

*Le numéro entre parenthèses sur chaque axe = fréquence dans les tirages filtrés.*

In [ ]:
figs = euroloto.plot_deep_companions(
    FIXED, kind=KIND,
    n_top=N_TOP, min_affinity=MIN_AFFINITY, metric=METRIC,
)
for fig in figs:
    display(fig)
    plt.close(fig)

---
## 4. Comparaison méthode Globale vs Filtrée

Cascade complète avec les deux méthodes côte à côte.

In [ ]:
result = euroloto.deep_companions(FIXED, kind=KIND, n_top=5, k_seeds=K_SEEDS)

---
## 5. Étoiles / Numéro chance

Distribution des numéros bonus dans les tirages contenant `FIXED`.

In [ ]:
_kind_b = 'euro' if KIND == 'all' else KIND
_df_b   = euroloto.draws(_kind_b)
_cfg_b  = GAMES[_kind_b]
_bonus_cols  = _cfg_b['bonus_cols']
_bonus_label = 'Étoiles' if _kind_b == 'euro' else 'Numéro chance'
b_min, b_max = _cfg_b['bonus_range']

# Filter draws containing FIXED
_mask = pd.Series([True] * len(_df_b), index=_df_b.index)
for _n in FIXED:
    _mask &= (_df_b[_cfg_b['main_cols']] == _n).any(axis=1)
_filt_b = _df_b[_mask]
n_filt  = len(_filt_b)

print(f"{n_filt} tirages {_cfg_b['name']} contenant {FIXED}")

if n_filt == 0:
    print("Aucun tirage trouvé.")
else:
    _bonus_vals = pd.concat([_filt_b[c] for c in _bonus_cols]).value_counts().sort_index()
    _full_bonus = _bonus_vals.reindex(range(b_min, b_max + 1), fill_value=0)
    _mean_b = _full_bonus.mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#e74c3c' if v > _mean_b * 1.1 else
              '#3498db' if v < _mean_b * 0.9 else '#95a5a6'
              for v in _full_bonus]
    ax.bar(_full_bonus.index, _full_bonus.values, color=colors, edgecolor='white')
    ax.axhline(_mean_b, color='#f39c12', linestyle='--', linewidth=1.8,
               label=f'Moyenne : {_mean_b:.1f}')
    ax.set_xlabel(_bonus_label, fontsize=12)
    ax.set_ylabel(f'Fréquence ({n_filt} tirages)', fontsize=12)
    ax.set_title(
        f'{_cfg_b["name"]} — {_bonus_label} dans les tirages contenant {FIXED}',
        fontsize=13, fontweight='bold',
    )
    ax.legend(fontsize=10)
    plt.tight_layout()

In [ ]:
# Co-occurrence des étoiles entre elles (EuroMillions seulement)
if _kind_b == 'euro' and n_filt >= 3 and len(_bonus_cols) > 1:
    _ecooc = cooccurrence_matrix(_filt_b, _bonus_cols).astype(float)
    np.fill_diagonal(_ecooc.values, np.nan)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        _ecooc, annot=True, fmt='.0f', cmap='Blues', ax=ax,
        mask=np.isnan(_ecooc), linewidths=0.5,
        cbar_kws={'label': f'Co-occurrences ({n_filt} tirages)', 'shrink': 0.8},
        annot_kws={'size': 9},
    )
    ax.set_title(
        f'Co-occurrences des Étoiles dans les tirages contenant {FIXED}\n'
        f'({n_filt} tirages)',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
else:
    print(f"Pas de heatmap étoiles : KIND='{_kind_b}' ou moins de 3 tirages.")